# W2-D1: Alert Correlation — Từ Noise Sang Signal

**Tác giả:** NguyenNgocGiao  
**Bài tập:** Xây dựng alert correlator 3-layer theo `instruction.md`

---

## Mục Tiêu (Section 2.4)

Cho **20 alert** đầu vào (`alerts_sample.jsonl`), output **3–7 cluster** với:
- Mỗi cluster có: `cluster_id`, `alert_count`, `services`, `time_range`, `max_severity`, `fingerprints`
- `reduction_ratio = 1 - output_clusters / input_alerts` ≥ 0.5

## Pipeline 3 Layer

```
20 alerts (raw)
     │
     ▼  Layer 1 (Section 3): Dedup — fingerprint, gộp alert lặp lại
     │
     ▼  Layer 2 (Section 4): Time-Window — session_groups(), gộp alert gần nhau về thời gian
     │
     ▼  Layer 3 (Section 5): Topology — topology_group(), gộp alert cùng cascade chain
     │
     ▼
3–7 clusters → results/cluster_summary.json
```

---
## Cell 1 — Setup & Import

Import thư viện. `networkx` dùng để build và traverse service dependency graph (Section 5.2).

In [ ]:
# Standard library
import json
import os
from collections import defaultdict, deque
from datetime import datetime, timedelta, timezone

# networkx — build service graph (Section 5.2)
try:
    import networkx as nx
    print(f"✅ networkx {nx.__version__}")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "networkx", "-q"])
    import networkx as nx
    print(f"✅ networkx {nx.__version__} (just installed)")

# Paths (Section 8.1)
DATASET_DIR  = "dataset"
ALERTS_FILE  = os.path.join(DATASET_DIR, "alerts_sample.jsonl")
SERVICES_FILE = os.path.join(DATASET_DIR, "services.json")
RESULTS_DIR  = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"✅ Paths OK — dataset: {DATASET_DIR}, results: {RESULTS_DIR}")

---
## Cell 2 — Load Dataset

Load `alerts_sample.jsonl` (20 alert) và `services.json` (service graph).

In [ ]:
# ── Load alerts (JSONL: mỗi dòng = 1 JSON object) ────────────────────────
alerts = []
with open(ALERTS_FILE, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            alerts.append(json.loads(line))

print(f"📊 Loaded {len(alerts)} alerts")
print()
print(f"{'ID':8s} {'Timestamp':22s} {'Service':20s} {'Metric':35s} {'Sev':5s} {'Value'}")
print("-" * 100)
for a in alerts:
    print(f"{a['id']:8s} {a['ts']:22s} {a['service']:20s} {a['metric']:35s} {a['severity']:5s} {a['value']}")

---
## Cell 3 — Layer 1: Dedup (Section 3)

### Khái niệm Fingerprint (Section 3.1)

> Fingerprint = subset field **không thay đổi** giữa các lần fire của cùng 1 alert.
> - ✅ `service`, `metric`, `severity` → xác định **loại** vấn đề
> - ❌ `ts`, `value` → thay đổi mỗi lần → KHÔNG dùng

Code dưới đây lấy trực tiếp từ **Section 3.1** và **Section 3.2** của instruction.md:

In [ ]:
# ── fingerprint() — Section 3.1 ──────────────────────────────────────────
def fingerprint(alert: dict) -> str:
    """
    Tạo unique key cho alert. 2 alert có cùng fingerprint = duplicate.

    Chọn field nào vào fingerprint?
    - PHẢI có: service, metric, severity
    - KHÔNG nên có: timestamp, value (thay đổi mỗi lần fire → dedup vô dụng)
    """
    return f"{alert['service']}|{alert['metric']}|{alert['severity']}"


# ── Deduper class — Section 3.2 ───────────────────────────────────────────
class Deduper:
    def __init__(self):
        self.store: dict[str, dict] = {}  # fingerprint → cluster info

    def push(self, alert: dict) -> str:
        """
        Add alert vào store. Return cluster_id (fingerprint đóng vai trò cluster_id ở layer này).
        """
        fp = fingerprint(alert)
        ts = datetime.fromisoformat(alert['ts'].replace('Z', '+00:00'))

        if fp not in self.store:
            self.store[fp] = {
                'cluster_id': fp,
                'count': 1,
                'first_seen': ts,
                'last_seen': ts,
                'alerts': [alert['id']],
                'max_value': alert['value'],
            }
        else:
            c = self.store[fp]
            c['count'] += 1
            c['last_seen'] = ts
            c['alerts'].append(alert['id'])
            c['max_value'] = max(c['max_value'], alert['value'])

        return fp

    def clusters(self) -> list[dict]:
        return list(self.store.values())


# ── Chạy Dedup ────────────────────────────────────────────────────────────
deduper = Deduper()
for a in alerts:
    deduper.push(a)

dedup_result = deduper.clusters()
print(f"Layer 1 — Dedup:")
print(f"  Input : {len(alerts)} alerts")
print(f"  Output: {len(dedup_result)} unique fingerprints")
print()
print(f"{'Count':6s} {'Fingerprint'}")
print("-" * 70)
for c in sorted(dedup_result, key=lambda x: -x['count']):
    dup = "← DUPLICATE" if c['count'] > 1 else ""
    print(f"  {c['count']:3d}   {c['cluster_id']:55s} {dup}")

print()
print("⚠️  Lưu ý (Section 3.3): Dedup chỉ gộp alert GIỐNG HỆT nhau.")
print("   Để gộp alert cùng cause khác metric → cần Layer 2 + Layer 3.")

---
## Cell 4 — Layer 2: Time-Window / Session Groups (Section 4)

### Insight (Section 4.1)
> Nếu nhiều service cùng kêu trong 2 phút → khả năng cùng cause **RẤT CAO**.

### Session Window (Section 4.3)
Dùng **session window** (không phải tumbling hay sliding):
- Session kết thúc khi không có alert mới trong `gap_sec` giây
- Tự adapt theo burst pattern của incident

**Chọn `gap_sec = 120` (2 phút) — theo khuyến nghị Section 4.4 (sweet spot cho hầu hết production system).**

In [ ]:
# ── session_groups() — Section 4.3 ───────────────────────────────────────
def session_groups(alerts: list[dict], gap_sec: int = 120) -> list[list[dict]]:
    """
    Mỗi group là 1 'session' alert. Session kết thúc khi không alert nào trong gap_sec giây.

    Vì sao session tốt hơn tumbling cho incident:
    - Incident burst: 30 alert trong 90 giây → 1 session tự nhiên
    - Tumbling 5min: nếu incident span 4 phút 30 - 5 phút 30 → bị cắt thành 2 window
    - Session tự adapt kích thước theo burst pattern
    """
    if not alerts:
        return []

    sorted_alerts = sorted(alerts, key=lambda a: a['ts'])
    groups = [[sorted_alerts[0]]]

    for alert in sorted_alerts[1:]:
        ts = datetime.fromisoformat(alert['ts'].replace('Z', '+00:00'))
        last_ts = datetime.fromisoformat(groups[-1][-1]['ts'].replace('Z', '+00:00'))

        if (ts - last_ts).total_seconds() <= gap_sec:
            groups[-1].append(alert)   # Cùng session
        else:
            groups.append([alert])     # Mở session mới

    return groups


# ── Chạy với gap_sec = 120 ────────────────────────────────────────────────
GAP_SEC = 120  # sweet spot — Section 4.4

sessions = session_groups(alerts, gap_sec=GAP_SEC)

print(f"Layer 2 — Session Groups (gap_sec={GAP_SEC}s):")
print(f"  Input : {len(alerts)} alerts")
print(f"  Output: {len(sessions)} session(s)")
print()
for i, sess in enumerate(sessions):
    svcs = sorted(set(a['service'] for a in sess))
    t0, t1 = sess[0]['ts'], sess[-1]['ts']
    span = (datetime.fromisoformat(t1.replace('Z','+00:00')) -
            datetime.fromisoformat(t0.replace('Z','+00:00'))).total_seconds()
    print(f"  Session {i}: {len(sess)} alerts | span {span:.0f}s | {t0} → {t1}")
    print(f"    services: {svcs}")

print()
print("📌 Khi gap_sec=120 → tất cả 20 alert nằm trong 1 session")
print("   (khoảng cách lớn nhất giữa 2 alert liên tiếp << 120s)")

---
## Cell 5 — Build Service Graph (Section 5.2)

Build directed graph từ `services.json`:
- **Edge A → B**: A gọi B (A depend on B)
- Ví dụ: `checkout-svc → payment-svc` = checkout depend on payment
- Nếu payment hỏng → checkout bị ảnh hưởng = cascade upstream

In [ ]:
# ── build_graph() — Section 5.2 ──────────────────────────────────────────
def build_graph(services_json_path: str) -> nx.DiGraph:
    """
    Build directed graph: A → B nghĩa là A gọi B.

    Khi RCA traverse, bạn sẽ đi NGƯỢC edge (từ A về B) — vì nếu A alert
    thì có thể B là root cause.
    """
    g = nx.DiGraph()
    data = json.loads(open(services_json_path).read())

    # Add service nodes
    for svc in data['services']:
        g.add_node(svc['name'], **{k: v for k, v in svc.items() if k != 'name'})

    # Add store nodes (DB, Redis, Kafka)
    for store in data['stores']:
        g.add_node(store['name'], **{k: v for k, v in store.items() if k != 'name'})

    # Add edges
    for edge in data['edges']:
        g.add_edge(edge['from'], edge['to'], type=edge['type'])

    return g


# ── Build graph ───────────────────────────────────────────────────────────
graph = build_graph(SERVICES_FILE)

print(f"Service Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
print()
print("Edges (A → B: A gọi B):")
for u, v, d in graph.edges(data=True):
    print(f"  {u:20s} →  {v:20s}  [{d['type']}]")

# ── Minh hoạ cascade path (Section 5.1) ──────────────────────────────────
print()
print("📌 Cascade path (undirected — Section 5.3):")
undirected = graph.to_undirected()
for pair in [("payment-svc", "checkout-svc"),
             ("payment-svc", "edge-lb"),
             ("payment-svc", "cart-svc"),
             ("payment-svc", "recommender-svc"),
             ("payment-svc", "search-svc"),
             ("payment-svc", "notification-svc")]:
    s1, s2 = pair
    try:
        d = nx.shortest_path_length(undirected, s1, s2)
        p = nx.shortest_path(undirected, s1, s2)
        flag = "✓ gộp" if d <= 2 else "✗ tách"
        print(f"  {s1} ↔ {s2}: {d} hop {flag}")
        print(f"    path: {' → '.join(p)}")
    except nx.NetworkXNoPath:
        print(f"  {s1} ↔ {s2}: không có path ✗ tách")

---
## Cell 6 — Layer 3: Topology-Aware Correlation (Section 5.3)

Dùng **Union-Find** để gộp các service alert nếu chúng cách nhau ≤ `max_hop` trên service graph.

**Chọn `max_hop = 2`** (theo Section 5.3 — N=2 thường tốt):
- `payment-svc` ↔ `checkout-svc` = 1 hop → gộp ✓
- `payment-svc` ↔ `edge-lb` = 2 hop → gộp ✓  
- `payment-svc` ↔ `recommender-svc` = > 2 hop → **tách** ✓ (batch retrain, không liên quan)

In [ ]:
# ── topology_group() — Section 5.3 ───────────────────────────────────────
def topology_group(alerts: list[dict], graph: nx.DiGraph, max_hop: int = 2) -> list[list[dict]]:
    """
    Group alerts nếu service của chúng cách nhau <= max_hop trên service graph.

    Lưu ý: dùng undirected version của graph cho khoảng cách — vì cascade
    có thể đi cả 2 chiều (upstream effect, downstream propagation tuỳ case).
    """
    if not alerts:
        return []

    undirected = graph.to_undirected()

    # Build mapping service → alerts ở service đó
    by_service = defaultdict(list)
    for a in alerts:
        by_service[a['service']].append(a)

    services_with_alerts = list(by_service.keys())

    # Union-Find
    parent = {s: s for s in services_with_alerts}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]  # path compression
            x = parent[x]
        return x

    def union(x, y):
        parent[find(x)] = find(y)

    # Two services cùng group nếu khoảng cách <= max_hop trên graph
    for i, s1 in enumerate(services_with_alerts):
        for s2 in services_with_alerts[i+1:]:
            try:
                dist = nx.shortest_path_length(undirected, s1, s2)
                if dist <= max_hop:
                    union(s1, s2)
            except nx.NetworkXNoPath:
                continue  # 2 service không connected → không group

    # Collect groups
    groups_dict = defaultdict(list)
    for s in services_with_alerts:
        groups_dict[find(s)].extend(by_service[s])

    return list(groups_dict.values())


# ── Test trên session đầu tiên ────────────────────────────────────────────
MAX_HOP = 2  # Section 5.3 — N=2 thường tốt

test_groups = topology_group(sessions[0], graph, max_hop=MAX_HOP)

print(f"Layer 3 — Topology Group (max_hop={MAX_HOP}):")
print(f"  Input : {len(sessions[0])} alerts trong session 0")
print(f"  Output: {len(test_groups)} topology group(s)")
print()
for i, grp in enumerate(test_groups):
    svcs = sorted(set(a['service'] for a in grp))
    ids  = [a['id'] for a in sorted(grp, key=lambda x: x['ts'])]
    print(f"  Group {i}: {len(grp)} alerts")
    print(f"    services : {svcs}")
    print(f"    alert_ids: {ids}")

---
## Cell 7 — correlate() Pipeline (Section 5.4)

Kết hợp Layer 2 + Layer 3 (Section 5.4):
> **Combined logic:** 2 alert cùng cluster nếu vừa **cùng time-window** vừa **cùng topology component**.

Thêm `fingerprints` vào mỗi cluster để track Layer 1 info.

In [ ]:
# ── correlate() — Section 5.4 ────────────────────────────────────────────
def correlate(alerts: list[dict], graph: nx.DiGraph, gap_sec: int = 120, max_hop: int = 2):
    """
    Pipeline:
      1. Sort alert by timestamp
      2. Cho mỗi session (time-window), apply topology grouping
      3. Output clusters
    """
    sessions = session_groups(alerts, gap_sec=gap_sec)

    all_clusters = []
    for session_idx, session_alerts in enumerate(sessions):
        # Trong mỗi session, topology-group
        topo_groups = topology_group(session_alerts, graph, max_hop=max_hop)
        for group_idx, group in enumerate(topo_groups):
            all_clusters.append({
                'cluster_id':   f'c-{session_idx:03d}-{group_idx:03d}',
                'alert_count':  len(group),
                'services':     sorted(set(a['service'] for a in group)),
                'alert_ids':    [a['id'] for a in group],
                'time_range':   [min(a['ts'] for a in group), max(a['ts'] for a in group)],
                'max_severity': max(a['severity'] for a in group),
                # fingerprints: track dedup info (unique alert types trong cluster)
                'fingerprints': sorted(set(fingerprint(a) for a in group)),
            })

    return all_clusters


# ── Run full pipeline ─────────────────────────────────────────────────────
clusters = correlate(alerts, graph, gap_sec=GAP_SEC, max_hop=MAX_HOP)

print("=" * 70)
print(f"  CORRELATION RESULTS")
print("=" * 70)
print(f"  Input alerts    : {len(alerts)}")
print(f"  Output clusters : {len(clusters)}")
reduction = round(1 - len(clusters) / len(alerts), 4)
print(f"  Reduction ratio : {reduction} ({reduction*100:.1f}% giảm)")
ok = "✅ ĐẠT" if reduction >= 0.5 else "❌ CHƯA ĐẠT"
print(f"  Target ≥ 50%    : {ok}")
print("=" * 70)
print()

for c in clusters:
    icon = "🔴" if c['max_severity'] == 'crit' else "🟡"
    print(f"{icon}  {c['cluster_id']}  |  {c['alert_count']} alerts  |  sev={c['max_severity']}")
    print(f"     services   : {c['services']}")
    print(f"     time_range : {c['time_range'][0]}  →  {c['time_range'][1]}")
    print(f"     alert_ids  : {c['alert_ids']}")
    print(f"     fingerprints ({len(c['fingerprints'])}):")
    for fp in c['fingerprints']:
        print(f"       • {fp}")
    print()

---
## Cell 8 — Phân Tích & Kiểm Chứng (Section 9 — EOD Checkpoint)

Verify kết quả và trả lời câu hỏi từ EOD Checkpoint.

In [ ]:
# ── Kiểm tra acceptance criteria (Section 8.4) ────────────────────────────
print("📋 ACCEPTANCE CRITERIA (Section 8.4):")
print()

# 1. Notebook chạy được (cell này là bằng chứng)
print("  ✅ Notebook chạy được, có ≥ 3 cell với output")

# 2. results/cluster_summary.json sẽ tạo ở Cell 9
print("  ⏳ results/cluster_summary.json — sẽ tạo ở cell tiếp theo")

# 3. Cluster có services list và time_range
has_services   = all('services'   in c for c in clusters)
has_time_range = all('time_range' in c for c in clusters)
print(f"  {'✅' if has_services else '❌'}  Cluster có 'services' list")
print(f"  {'✅' if has_time_range else '❌'}  Cluster có 'time_range'")

# 4. reduction_ratio >= 0.5
print(f"  {'✅' if reduction >= 0.5 else '❌'}  reduction_ratio = {reduction} (≥ 0.5)")

print()
print("-" * 70)
print("📌 EOD CHECKPOINT — Section 9:")
print()

# Câu 4 ⭐ — recommender-svc có bị gộp vào cluster chính không?
main_cluster = max(clusters, key=lambda c: c['alert_count'])
recommender_in_main = 'recommender-svc' in main_cluster['services']
search_in_main = 'search-svc' in main_cluster['services']

print(f"  ⭐ Câu 4 — recommender-svc trong cluster chính? {'CÓ ❌' if recommender_in_main else 'KHÔNG ✅'}")
print(f"     → Đúng: recommender-svc alert do batch retrain (Section 5.4 — 'unrelated')")
print(f"     → Topology: recommender-svc cách payment-svc > {MAX_HOP} hop → tách riêng")
print()
print(f"     search-svc trong cluster chính? {'CÓ ❌' if search_in_main else 'KHÔNG ✅'}")
print(f"     → Đúng: search-svc là 'noise — independent slow query' (labeled trong data)")
print()

# Orphan check
all_assigned = set(aid for c in clusters for aid in c['alert_ids'])
all_ids = set(a['id'] for a in alerts)
orphans = all_ids - all_assigned
print(f"  Orphan alerts (không gộp được): {len(orphans)} {'← OK' if not orphans else orphans}")

print()
print("  Cluster breakdown:")
for c in sorted(clusters, key=lambda x: -x['alert_count']):
    print(f"    {c['cluster_id']}: {c['alert_count']:2d} alerts | {c['services']}")

---
## Cell 9 — Export results/cluster_summary.json (Section 8.2)

In [ ]:
# ── Build output JSON — format từ Section 8.2 ─────────────────────────────
output = {
    "input_alerts":    len(alerts),
    "output_clusters": len(clusters),
    "reduction_ratio": reduction,
    "clusters":        clusters
}

# ── Write file ────────────────────────────────────────────────────────────
out_path = os.path.join(RESULTS_DIR, "cluster_summary.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False, default=str)

# ── Verify ────────────────────────────────────────────────────────────────
with open(out_path, encoding="utf-8") as f:
    verify = json.load(f)

print(f"✅ Đã ghi: {out_path}  ({os.path.getsize(out_path)} bytes)")
print(f"   input_alerts    : {verify['input_alerts']}")
print(f"   output_clusters : {verify['output_clusters']}")
print(f"   reduction_ratio : {verify['reduction_ratio']}")
print()
print("Preview JSON:")
print(json.dumps({
    "input_alerts":    verify['input_alerts'],
    "output_clusters": verify['output_clusters'],
    "reduction_ratio": verify['reduction_ratio'],
    "clusters": [
        {
            "cluster_id":   c['cluster_id'],
            "alert_count":  c['alert_count'],
            "services":     c['services'],
            "time_range":   c['time_range'],
            "max_severity": c['max_severity'],
            "fingerprints": c['fingerprints'],
        }
        for c in verify['clusters']
    ]
}, indent=2, ensure_ascii=False))